# 01 · Phase 1 — Download and validate the raw data

**Goal.** Put the four pinned open datasets (CISA KEV, NVD, MITRE ATT&CK 16.1, CTID PKEV) into `data/raw/`
and prove they are *usable* before any later phase starts.

**Phase gate** (all five must pass, otherwise we stop):

| # | Check | Question it answers |
|---|-------|---------------------|
| 1 | Existence | Is every expected file there (and nothing extra)? |
| 2 | Integrity | Are the bytes identical to the pinned version (SHA-256)? |
| 3 | Format | Does the content parse and have the expected structure and counts? |
| 4 | Timestamps | Do the dates match the common version (2025-07-28)? |
| 5 | Accessibility | Can the files be read (and are they protected from edits)? |

Code lives in `src/threatgraph/`; this notebook only *calls* it and shows the results.
Registry of URLs/commits/hashes: `config/sources.yaml`.

In [1]:
# Bootstrap: find the repo root (works wherever Jupyter was started) and make `src/` importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sources.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
%load_ext autoreload
%autoreload 2
print("project root:", ROOT)

project root: /home/tavaa/Advanced_data_management/ThreatGraph


In [2]:
import json, shutil, tempfile, platform
import pandas as pd
import requests
from IPython.display import display

from threatgraph import config
from threatgraph.download import (make_session, preflight, download_core_files, load_kev_cve_ids,
                                  download_nvd_records, nvd_relpath, core_file_specs)
from threatgraph.manifest import write_manifest, read_manifest, records_by_relpath
from threatgraph import validate_raw as V
from threatgraph import peek

SRC = config.load_sources()
RAW = config.RAW_DIR
print("python", platform.python_version(), "| requests", requests.__version__, "| pandas", pd.__version__)
print("registry digest:", config.sources_digest()[:16], "...")

python 3.11.17 | requests 2.34.2 | pandas 3.0.6
registry digest: c14c1352221bdbf8 ...


## 1 · The registry
Every source is pinned to a **git commit** (not a branch): the same URL gives the same bytes tomorrow.

In [3]:
rows = [{"source": s["source"], "role": s["role"], "file": s["relpath"], "pinned_ref": s["pinned_ref"][:10],
         "expected_bytes": s["bytes"], "expected_sha256": s["sha256"][:12] + "…"} for s in core_file_specs(SRC)]
display(pd.DataFrame(rows))
print("NVD per-CVE rule:", SRC["sources"]["nvd"]["per_cve"]["path_template"])
print("example:", nvd_relpath("CVE-2024-34102"), "|", nvd_relpath("CVE-2023-2533"))
assert nvd_relpath("CVE-2024-34102") == "CVE-2024/CVE-2024-341xx/CVE-2024-34102.json"
assert nvd_relpath("CVE-2023-2533") == "CVE-2023/CVE-2023-25xx/CVE-2023-2533.json"

,source,role,file,pinned_ref,expected_bytes,expected_sha256
0,kev,data,kev/known_exploited_vulnerabilities.json,0cca62ed46,1243169,b20607afbdda…
1,kev,schema,kev/known_exploited_vulnerabilities_schema.json,0cca62ed46,3407,577f4ccc06b7…
2,kev,license,kev/LICENSE,0cca62ed46,7469,8dfcf2d0cba3…
3,attack,data,attack/enterprise-attack-16.1.json,6cda5ad846,46485155,8423d8dac3fc…
4,attack,license,attack/LICENSE.txt,6cda5ad846,1354,738144f7fb05…
5,pkev,data,pkev/kev-07.28.2025_attack-16.1-enterprise.json,ddc842aa48,1210128,b57b3d382809…
6,pkev,schema,pkev/mapex-unified-data-schema.json,ddc842aa48,7384,1ecd1437bde6…
7,pkev,license,pkev/LICENSE.txt,ddc842aa48,10174,0d542e0c8804…
8,nvd,license,nvd/nvd-tou.md,a4262227a2,3444,b1040b68c3c7…
9,nvd,license,nvd/cve-tou.md,a4262227a2,1729,f04031049349…


NVD per-CVE rule: CVE-{year}/CVE-{year}-{bucket}xx/{cve_id}.json
example: CVE-2024/CVE-2024-341xx/CVE-2024-34102.json | CVE-2023/CVE-2023-25xx/CVE-2023-2533.json


## 2 · Preflight (online accessibility)
A cheap `HEAD` request per URL: fail now, not after downloading 46 MB.

In [4]:
pf = preflight(SRC)
display(pd.DataFrame(pf)[["label", "status", "content_length"]])
assert all(r["status"] == 200 for r in pf), "some pinned URL is not reachable"

,label,status,content_length
0,kev/known_exploited_vulnerabilities.json,200,1243169
1,kev/known_exploited_vulnerabilities_schema.json,200,3407
2,kev/LICENSE,200,7469
3,attack/enterprise-attack-16.1.json,200,46485155
4,attack/LICENSE.txt,200,1354
5,pkev/kev-07.28.2025_attack-16.1-enterprise.json,200,1210128
6,pkev/mapex-unified-data-schema.json,200,7384
7,pkev/LICENSE.txt,200,10174
8,nvd/nvd-tou.md,200,3444
9,nvd/cve-tou.md,200,1729


## 3 · Download the core files (KEV, ATT&CK, PKEV + schemas + licenses)
Idempotent: a file that is already present **and** has the pinned SHA-256 is not downloaded again (`cached`).

In [5]:
previous = records_by_relpath(read_manifest(config.MANIFEST_PATH)) if config.MANIFEST_PATH.exists() else {}
core_records = download_core_files(SRC, RAW, previous=previous)
assert len(core_records) == len(core_file_specs(SRC))

  [cached    ] kev/known_exploited_vulnerabilities.json                   1,243,169 B
  [cached    ] kev/known_exploited_vulnerabilities_schema.json                3,407 B
  [cached    ] kev/LICENSE                                                    7,469 B
  [cached    ] attack/enterprise-attack-16.1.json                        46,485,155 B
  [cached    ] attack/LICENSE.txt                                             1,354 B
  [cached    ] pkev/kev-07.28.2025_attack-16.1-enterprise.json            1,210,128 B
  [cached    ] pkev/mapex-unified-data-schema.json                            7,384 B
  [cached    ] pkev/LICENSE.txt                                              10,174 B
  [cached    ] nvd/nvd-tou.md                                                 3,444 B
  [cached    ] nvd/cve-tou.md                                                 1,729 B


In [6]:
# Idempotency proof: a second call must download nothing.
again = download_core_files(SRC, RAW, previous=records_by_relpath(read_manifest(config.MANIFEST_PATH)) if config.MANIFEST_PATH.exists() else {}, log=lambda *_: None)
assert all(r.status == "cached" for r in again), "second run should hit the cache"
print("second run: all", len(again), "files cached")

second run: all 10 files cached


## 4 · NVD: one JSON per KEV CVE
The project scope is *the CVEs listed in KEV*. The list is **derived from the downloaded KEV file**, not typed by hand.

In [7]:
cve_ids = load_kev_cve_ids(RAW / "kev" / "known_exploited_vulnerabilities.json")
print(len(cve_ids), "CVE ids in KEV; first 3:", cve_ids[:3])
assert len(cve_ids) == SRC["expected_counts"]["kev_entries"]

nvd_records, nvd_errors = download_nvd_records(cve_ids, SRC, RAW, workers=8, previous=previous)
print("errors:", len(nvd_errors), "| downloaded:", sum(r.status == "downloaded" for r in nvd_records),
      "| cached:", sum(r.status == "cached" for r in nvd_records))
assert nvd_errors == {}, nvd_errors
assert len(nvd_records) == len(cve_ids)

1391 CVE ids in KEV; first 3: ['CVE-2002-0367', 'CVE-2004-0210', 'CVE-2004-1464']


NVD records:   0%|          | 0/1391 [00:00<?, ?file/s]

errors: 0 | downloaded: 0 | cached: 1391


## 5 · Manifest (provenance of every file)

In [8]:
meta = {"project": "threatgraph-ld", "phase": 1, "sources_digest_sha256": config.sources_digest(),
        "common_version": SRC["common_version"],
        "environment": {"python": platform.python_version(), "requests": requests.__version__}}
manifest = write_manifest(config.MANIFEST_PATH, core_records + nvd_records, meta)
print("manifest:", config.MANIFEST_PATH.relative_to(ROOT), "|", manifest["file_count"], "files |", f"{manifest['total_bytes']/2**20:.1f} MiB")
display(pd.DataFrame(manifest["files"]).drop(columns=["etag", "license", "expected_sha256"]).head(6))

manifest: data/raw/manifest.json | 1401 files | 63.3 MiB


,source,role,relpath,url,bytes,sha256,retrieved_at_utc,http_status,status,pinned_ref
0,attack,license,attack/LICENSE.txt,https://raw.githubusercontent.com/mitre-attack...,1354,738144f7fb054722a4ef9d3367c51710341dc12fc574c6...,2026-10-13T13:44:14Z,200,cached,6cda5ad8462c79e14fbb872f4e09059b18e0cfc4
1,attack,data,attack/enterprise-attack-16.1.json,https://raw.githubusercontent.com/mitre-attack...,46485155,8423d8dac3fc2feb825bb07d26e5f5d905e08a88f6fe46...,2026-10-13T13:44:14Z,200,cached,6cda5ad8462c79e14fbb872f4e09059b18e0cfc4
2,kev,license,kev/LICENSE,https://raw.githubusercontent.com/cisagov/kev-...,7469,8dfcf2d0cba33bede386eb470daf2ac08caf6540fc2516...,2026-10-13T13:44:14Z,200,cached,0cca62ed469c333a45760beeaf35fcc53352aedd
3,kev,data,kev/known_exploited_vulnerabilities.json,https://raw.githubusercontent.com/cisagov/kev-...,1243169,b20607afbdda3f6393282e797f4e49f87b48f5b64f124a...,2026-10-13T13:44:14Z,200,cached,0cca62ed469c333a45760beeaf35fcc53352aedd
4,kev,schema,kev/known_exploited_vulnerabilities_schema.json,https://raw.githubusercontent.com/cisagov/kev-...,3407,577f4ccc06b7b7c6a109e1a0d6457a26db7fc5219398ff...,2026-10-13T13:44:14Z,200,cached,0cca62ed469c333a45760beeaf35fcc53352aedd
5,nvd,license,nvd/cve-tou.md,https://raw.githubusercontent.com/fkie-cad/nvd...,1729,f04031049349f83c5d6276d529b5529c62f7ade891896d...,2026-10-13T13:44:14Z,200,cached,a4262227a26a698199ea8d287932f8639e85aa6d


## 6 · Look at the data (formats)
Quick peeks so we *see* what each source looks like before trusting any check.

### 6.1 KEV — flat JSON list

In [9]:
kev = json.loads((RAW / "kev" / "known_exploited_vulnerabilities.json").read_text())
o = peek.kev_overview(kev)
print(o["header"]); print(peek.pretty(o["first_entry"]))
print("ransomware flag:", o["ransomware"], "| dateAdded range:", o["dateAdded_range"])
display(o["frame"][["cveID", "vendorProject", "product", "dateAdded", "knownRansomwareCampaignUse"]].head(5))

{'title': 'CISA Catalog of Known Exploited Vulnerabilities', 'catalogVersion': '2025.07.28', 'dateReleased': '2025-07-28T14:00:14.6746Z', 'count': 1391}
{
  "cveID": "CVE-2023-2533",
  "vendorProject": "PaperCut",
  "product": "NG/MF",
  "vulnerabilityName": "PaperCut NG/MF Cross-Site Request Forgery (CSRF) Vulnerability",
  "dateAdded": "2025-07-28",
  "shortDescription": "PaperCut NG/MF contains a cross-site request forgery (CSRF) vulnerability, which, under specific conditions, could potentially enable an attacker to alter security settings or execute arbitrary code. ",
  "requiredAction": "Apply mitigations per vendor instructions, follow applicable BOD 22-01 guidance for cloud services, or discontinue use of the product if mitigations are unavailable.",
  "dueDate": "2025-08-18",
  "knownRansomwareCampaignUse": "Unknown",
  "notes": "https://www.papercut.com/kb/Main/SecurityBulletinJune2023 ; https://nvd.nist.gov/vuln/detail/CVE-2023-2533",
  "cwes": [
    "CWE-352"
  ]
}
ransomwa

,cveID,vendorProject,product,dateAdded,knownRansomwareCampaignUse
0,CVE-2023-2533,PaperCut,NG/MF,2025-07-28,Unknown
1,CVE-2025-20337,Cisco,Identity Services Engine,2025-07-28,Unknown
2,CVE-2025-20281,Cisco,Identity Services Engine,2025-07-28,Unknown
3,CVE-2025-2775,SysAid,SysAid On-Prem,2025-07-22,Unknown
4,CVE-2025-2776,SysAid,SysAid On-Prem,2025-07-22,Unknown


### 6.2 PKEV — `metadata` + `mapping_objects` (one row per CVE–technique pair)

In [10]:
pkev = json.loads(next((RAW / "pkev").glob("kev-*.json")).read_text())
o = peek.pkev_overview(pkev)
print(peek.pretty(o["metadata"], 600)); print(peek.pretty(o["first_row"], 900))
print("mapping_type:", o["mapping_type"])
display(o["top_techniques"].rename("rows").to_frame())

{
  "mapping_version": "",
  "technology_domain": "enterprise",
  "attack_version": "16.1",
  "mapping_framework": "kev",
  "mapping_framework_version": "07/28/2025",
  "author": null,
  "contact": null,
  "organization": null,
  "creation_date": "07/28/2025",
  "last_update": "08/28/2025",
  "mapping_types": {
    "primary_impact": {
      "name": "primary_impact",
      "description": ""
    },
    "secondary_impact": {
      "name": "secondary_impact",
      "description": ""
    },
    "exploitation_technique": {
      "name": "exploitation_technique",
      "description": ""
    },
    "u
  ... [truncated]
{
  "capability_id": "CVE-2024-34102",
  "capability_description": "Adobe Commerce and Magento Open Source Improper Restriction of XML External Entity Reference (XXE) Vulnerability",
  "mapping_type": "exploitation_technique",
  "attack_object_id": "T1190",
  "attack_object_name": "Exploit Public-Facing Application",
  "capability_group": "xxe",
  "comments": "This vulnerability

,,rows
attack_object_id,attack_object_name,
T1059,Command and Scripting Interpreter,170
T1190,Exploit Public-Facing Application,157
T1068,Exploitation for Privilege Escalation,69
T1078,Valid Accounts,46
T1005,Data from Local System,46


### 6.3 ATT&CK — STIX 2.1 bundle (a graph of typed objects)

In [11]:
attack = json.loads(next((RAW / "attack").glob("enterprise-attack-*.json")).read_text())
o = peek.attack_overview(attack)
print(o["bundle"]); print(peek.pretty(o["sample_technique"], 700))
display(o["type_counts"].rename("objects").to_frame())

{'type': 'bundle', 'id': 'bundle--1b5de6a3-1a45-4f85-8103-81cff6d77e22', 'spec_version': '2.1'}
{
  "type": "attack-pattern",
  "id": "attack-pattern--0042a9f5-f053-4769-b3ef-9ad018dfa298",
  "name": "Extra Window Memory Injection",
  "kill_chain_phases": [
    {
      "kill_chain_name": "mitre-attack",
      "phase_name": "defense-evasion"
    },
    {
      "kill_chain_name": "mitre-attack",
      "phase_name": "privilege-escalation"
    }
  ],
  "modified": "2022-04-25T14:00:00.188Z",
  "external_id": "T1055.011"
}


,objects
relationship,20731
attack-pattern,799
malware,626
course-of-action,285
intrusion-set,176
x-mitre-data-component,109
tool,89
x-mitre-data-source,38
campaign,34
x-mitre-tactic,14


### 6.4 NVD — one record per CVE (NVD API 2.0 shape)

In [12]:
rec = json.loads((RAW / "nvd" / "cve" / "CVE-2024-34102.json").read_text())
o = peek.nvd_overview(rec)
print(peek.pretty(o["summary"], 600)); print("keys:", o["keys"]); print("metrics:", o["metrics_keys"]); print(o["description_en"])

{
  "id": "CVE-2024-34102",
  "sourceIdentifier": "psirt@adobe.com",
  "published": "2024-06-13T09:15:10.380",
  "lastModified": "2024-11-29T15:33:14.280",
  "vulnStatus": "Analyzed"
}
keys: ['cisaActionDue', 'cisaExploitAdd', 'cisaRequiredAction', 'cisaVulnerabilityName', 'configurations', 'cveTags', 'descriptions', 'id', 'lastModified', 'metrics', 'published', 'references', 'sourceIdentifier', 'vulnStatus', 'weaknesses']
metrics: ['cvssMetricV31']
Adobe Commerce versions 2.4.7, 2.4.6-p5, 2.4.5-p7, 2.4.4-p8 and earlier are affected by an Improper Restriction of XML External Entity Reference ('XXE') vulnerability that could result in arbitrary co...


## 7 · Validation — the five families
All checks run on the files **on disk**, not on variables in memory.

In [13]:
manifest = read_manifest(config.MANIFEST_PATH)          # re-read from disk on purpose
results, stats = V.run_all(RAW, SRC, manifest)
summary = pd.DataFrame(V.summarize(results)).T
summary["total"] = summary.passed + summary.failed
display(summary)
print("stats:", stats)
failed = [r for r in results if not r.ok]
print("FAILED:", failed if failed else "none")

,passed,failed,total
existence,25,0,25
integrity,33,0,33
format,27,0,27
timestamps,14,0,14
accessibility,13,11,24


stats: {'parse_seconds': {'kev': 0.003, 'pkev': 0.004, 'attack': 0.2, 'nvd_all': 0.151}, 'nvd_hash_seconds': 0.03, 'nvd_max_lastModified': '2025-07-23T18:10:23.830000+00:00', 'nvd_records_parsed': 1391}
FAILED: [CheckResult(family='accessibility', name='read-only mode kev/known_exploited_vulnerabilities.json', ok=False, detail='0o644'), CheckResult(family='accessibility', name='read-only mode kev/known_exploited_vulnerabilities_schema.json', ok=False, detail='0o644'), CheckResult(family='accessibility', name='read-only mode kev/LICENSE', ok=False, detail='0o644'), CheckResult(family='accessibility', name='read-only mode attack/enterprise-attack-16.1.json', ok=False, detail='0o644'), CheckResult(family='accessibility', name='read-only mode attack/LICENSE.txt', ok=False, detail='0o644'), CheckResult(family='accessibility', name='read-only mode pkev/kev-07.28.2025_attack-16.1-enterprise.json', ok=False, detail='0o644'), CheckResult(family='accessibility', name='read-only mode pkev/mapex-u

In [14]:
# Full list, one row per check, so nothing is hidden behind a green "OK".
df = pd.DataFrame([r.__dict__ for r in results])
pd.set_option("display.max_colwidth", 90, "display.max_rows", 200, "display.width", 200)
display(df[df.family == "timestamps"][["name", "ok", "detail"]])

,name,ok,detail
85,kev: catalogVersion == common version,True,2025.07.28
86,kev: dateReleased falls on the common-version day,True,2025-07-28T14:00:14.6746Z
87,kev: max(dateAdded) <= dateReleased,True,2025-07-28
88,kev: dueDate < dateAdded only in the known anomalies,True,"15 found, 15 known"
89,kev: dateReleased not in the future,True,
90,pkev: creation_date <= last_update,True,07/28/2025 <= 08/28/2025
91,pkev: dates not in the future,True,
92,pkev: attack_version == common version,True,
93,attack: collection x_mitre_version == common version,True,16.1
94,attack: all 'modified' parse and are not in the future,True,max modified = 2024-11-12T14:00:00.188000+00:00


### 7.1 Known anomalies in the source data
Two checks were first written as "no violations allowed" and **failed on the real data**. They are not bugs of this code:

* **KEV**: 15 entries with `dueDate` earlier than `dateAdded` (all added on 2021-11-03, the initial bulk load).
* **ATT&CK**: 2 `relationship` objects with `created` later than `modified`.

We keep the checks, but assert the *exact* number (`known_anomalies` in the registry). If the number changes, the data changed.

In [15]:
kev_bad = [(v["cveID"], v["dateAdded"], v["dueDate"]) for v in kev["vulnerabilities"] if v["dateAdded"] > v["dueDate"]]
print(len(kev_bad), "KEV entries with dueDate < dateAdded; dateAdded values:", {x[1] for x in kev_bad})
atk_bad = [o["id"] for o in attack["objects"] if "created" in o and "modified" in o and V.parse_iso(o["created"]) > V.parse_iso(o["modified"])]
print(len(atk_bad), "ATT&CK objects with created > modified:", atk_bad)
assert len(kev_bad) == SRC["known_anomalies"]["kev_due_before_added"]
assert len(atk_bad) == SRC["known_anomalies"]["attack_created_after_modified"]

15 KEV entries with dueDate < dateAdded; dateAdded values: {'2021-11-03'}
2 ATT&CK objects with created > modified: ['relationship--8eafd02c-0c86-4a7d-ae64-d3c6c04c9d2e', 'relationship--b184e04d-1d27-4fd4-a853-610629ec8694']


## 8 · Negative tests — do the checks really fail on bad data?
A check that has never failed is not proven. We damage **temporary copies** (the real files stay untouched).

In [16]:
tmp = Path(tempfile.mkdtemp(prefix="tg_neg_"))
kev_spec = next(s for s in core_file_specs(SRC) if s["role"] == "data" and s["source"] == "kev")
src_file = RAW / kev_spec["relpath"]
tests = []   # (description, detected?)

def detected(check_results):            # a negative test passes when at least one check FAILS
    return any(not r.ok for r in check_results)

# 1 bit flip -> integrity
f = tmp / "kev_flip.json"; shutil.copyfile(src_file, f)
b = bytearray(f.read_bytes()); b[1000] ^= 0x01; f.write_bytes(bytes(b))
tests.append(("1 flipped bit  -> sha256 mismatch", detected(V.check_file_integrity(f, kev_spec["sha256"], kev_spec["bytes"]))))
# 2 truncation -> integrity
f = tmp / "kev_trunc.json"; f.write_bytes(src_file.read_bytes()[:-500])
tests.append(("2 truncated file -> size/sha mismatch", detected(V.check_file_integrity(f, kev_spec["sha256"], kev_spec["bytes"]))))
# 3 missing file -> existence
tests.append(("3 missing file  -> existence", detected([V.check_file_exists(tmp / "nope.json")])))
# 4 empty file -> existence
(tmp / "empty.json").write_bytes(b"")
tests.append(("4 empty file    -> existence", detected([V.check_file_exists(tmp / "empty.json")])))
# 5 KEV entry removed -> format
bad = json.loads(json.dumps(kev)); bad["vulnerabilities"].pop()
tests.append(("5 KEV entry removed -> count mismatch", detected(V.check_kev_format(bad, SRC))))
# 6 KEV field removed -> format
bad = json.loads(json.dumps(kev)); del bad["vulnerabilities"][0]["notes"]
tests.append(("6 KEV field removed -> structure", detected(V.check_kev_format(bad, SRC))))
# 7 KEV wrong catalog version -> timestamps
bad = json.loads(json.dumps(kev)); bad["catalogVersion"] = "2025.08.01"
tests.append(("7 wrong catalogVersion -> timestamps", detected(V.check_kev_timestamps(bad, SRC))))
# 8 NVD record modified after the cutoff -> timestamps
bad = json.loads(json.dumps(rec)); bad["lastModified"] = "2026-01-01T00:00:00.000"
tests.append(("8 NVD lastModified after cutoff -> timestamps", detected(V.check_nvd_timestamps(bad, "CVE-2024-34102", SRC))))
# 9 NVD record with wrong id -> format
bad = json.loads(json.dumps(rec)); bad["id"] = "CVE-1999-0001"
tests.append(("9 NVD id != filename -> format", detected(V.check_nvd_record_format(bad, "CVE-2024-34102"))))
# 10 PKEV pointing to an unknown technique -> cross-reference
bad = json.loads(json.dumps(pkev)); bad["mapping_objects"][0]["attack_object_id"] = "T9999"
tests.append(("10 PKEV unknown technique -> x-ref", detected(V.check_pkev_format(bad, SRC, {r['capability_id'] for r in pkev['mapping_objects']}, V.attack_technique_ids(attack)))))
# 11 writable file -> accessibility (raw data must be read-only)
f = tmp / "writable.json"; shutil.copyfile(src_file, f); f.chmod(0o644)
tests.append(("11 writable file -> accessibility (read-only rule)", detected(V.check_file_accessible(f))))

neg = pd.DataFrame(tests, columns=["negative test", "detected"])
display(neg)
shutil.rmtree(tmp)
assert neg.detected.all(), "a check failed to detect a deliberately broken input"
print("all", len(neg), "negative tests detected")

,negative test,detected
0,1 flipped bit -> sha256 mismatch,True
1,2 truncated file -> size/sha mismatch,True
2,3 missing file -> existence,True
3,4 empty file -> existence,True
4,5 KEV entry removed -> count mismatch,True
5,6 KEV field removed -> structure,True
6,7 wrong catalogVersion -> timestamps,True
7,8 NVD lastModified after cutoff -> timestamps,True
8,9 NVD id != filename -> format,True
9,10 PKEV unknown technique -> x-ref,True


all 11 negative tests detected


## 9 · Phase gate
Only if every check passes do we write the report and allow Phase 2.

In [17]:
V.assert_all(results)                      # raises AssertionError listing every failed check
report = V.write_report(config.REPORT_PATH, results, stats,
                        {"manifest_sha256": V.sha256_file(config.MANIFEST_PATH), "sources_digest_sha256": config.sources_digest(),
                         "negative_tests_detected": int(neg.detected.sum()), "negative_tests_total": len(neg)})
print("PHASE 1 GATE: PASSED  ->", report["summary"])
print("report:", config.REPORT_PATH.relative_to(ROOT))

AssertionError: 11 validation check(s) failed:
  [accessibility] read-only mode kev/known_exploited_vulnerabilities.json: 0o644
  [accessibility] read-only mode kev/known_exploited_vulnerabilities_schema.json: 0o644
  [accessibility] read-only mode kev/LICENSE: 0o644
  [accessibility] read-only mode attack/enterprise-attack-16.1.json: 0o644
  [accessibility] read-only mode attack/LICENSE.txt: 0o644
  [accessibility] read-only mode pkev/kev-07.28.2025_attack-16.1-enterprise.json: 0o644
  [accessibility] read-only mode pkev/mapex-unified-data-schema.json: 0o644
  [accessibility] read-only mode pkev/LICENSE.txt: 0o644
  [accessibility] read-only mode nvd/nvd-tou.md: 0o644
  [accessibility] read-only mode nvd/cve-tou.md: 0o644
  [accessibility] nvd: all records read-only (mode bits): 1391 writable

## 10 · What this notebook does **not** prove
* The bytes are the ones **we pinned** — not that the upstream data is *correct* (see the 17 known anomalies).
* NVD comes from a **community mirror** (`fkie-cad`), not directly from NIST: [UNCERTAIN] fidelity beyond the checks above.
* `cisa.gov` / `nvd.nist.gov` were not queried: the pins are GitHub commits.
* PKEV data license: only the repo's Apache-2.0 file is present; no separate data license [UNCERTAIN].

**Next (Phase 2, conceptual only):** profile each source, define cleaning rules, write `data/clean/`.